In [1]:
import pandas as pd 
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
# from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.naive_bayes import MultinomialNB
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from sklearn.feature_extraction.text import TfidfVectorizer
import joblib

df = pd.read_csv("../datasets/reddit.csv")
df.rename(columns={'clean_text': 'message', 'is_depression': 'label'}, inplace=True)


In [2]:
X_train, X_test, y_train, y_test = train_test_split(df["message"], df['label'], test_size=0.3)
pipeline = Pipeline([
    ('tfidf', TfidfVectorizer()),
    ('classifier', MultinomialNB())
])
pipeline.fit(X_train, y_train)
y_pred = pipeline.predict(X_test)

classification_report_result = classification_report(y_test, y_pred)
print(classification_report_result)

              precision    recall  f1-score   support

           0       0.98      0.53      0.69      1163
           1       0.68      0.99      0.81      1157

    accuracy                           0.76      2320
   macro avg       0.83      0.76      0.75      2320
weighted avg       0.83      0.76      0.75      2320



In [3]:
from xgboost import XGBClassifier


xgb_model = XGBClassifier(use_label_encoder=False, eval_metric='logloss')
xgb_pipeline = Pipeline([
    ('tfidf', TfidfVectorizer()),
    ('classifier', xgb_model)
])
xgb_pipeline.fit(X_train, y_train)
y_pred_xgb = xgb_pipeline.predict(X_test)

classification_report_result = classification_report(y_test, y_pred_xgb)
print(classification_report_result)

/home/hyo/Documents/projects/trackerv2/.venv/lib/python3.13/site-packages/xgboost/training.py:183: UserWarning: [20:37:29] WARNING: /workspace/src/learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


              precision    recall  f1-score   support

           0       0.95      0.99      0.97      1163
           1       0.99      0.94      0.97      1157

    accuracy                           0.97      2320
   macro avg       0.97      0.97      0.97      2320
weighted avg       0.97      0.97      0.97      2320



In [4]:
from lightgbm import LGBMClassifier
lgbm_model = LGBMClassifier(n_jobs=-1, reg_lambda=0.1, num_leaves=64, max_depth=-1, learning_rate=0.1, n_estimators=100)
lgbm_pipeline = Pipeline([
    ('tfidf', TfidfVectorizer()),
    ('classifier', lgbm_model)
])
lgbm_pipeline.fit(X_train, y_train)
y_pred_lgbm = lgbm_pipeline.predict(X_test)
classification_report_result = classification_report(y_test, y_pred_lgbm)
print(classification_report_result)

[LightGBM] [Info] Number of positive: 2674, number of negative: 2737
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.044816 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 55495
[LightGBM] [Info] Number of data points in the train set: 5411, number of used features: 1341
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.494179 -> initscore=-0.023287
[LightGBM] [Info] Start training from score -0.023287
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
              precision    recall  f1-score   support

           0       0.94      0.99      0.97      1163
           1       0.99      0.94      0.96      1157

    accuracy                           0.97      2320
   macro avg       0.97      0.97      0.97      2320
weighted avg       0.97      0.97      0.97      2320



/home/hyo/Documents/projects/trackerv2/.venv/lib/python3.13/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


In [5]:
df.shape

(7731, 2)

In [7]:
df.groupby(df["label"]).count()

,message
label,
0,3900
1,3831


In [9]:
df_only_depression = df[df['label'] == 1].copy()
if not df_only_depression.empty:
    df_only_depression['label'] = 'depression'

df_only_depression.to_csv('depression_only.csv', index=False)